# 02 — Connecting to MLflow

MLflow tracks your machine learning experiments — it records parameters, metrics, and results so you can compare runs over time.

Run each cell from top to bottom by pressing **Shift + Enter**.

In [ ]:
import os
os.environ["UV_EXTRA_INDEX_URL"] = "https://pypi.org/simple"

# enable colour output for all print() calls in this notebook
!uv pip install rich -q
from rich import print, print_json

## Step 1 — Install MLflow

In [ ]:
!uv pip install "mlflow[kubernetes]" openai -q

## Step 2 — Get an auth token

MLflow on OpenShift requires a bearer token. The cell below tries the mounted ServiceAccount token first (automatically available inside a notebook pod), then falls back to `oc whoami -t`.

In [ ]:
import os
import subprocess

SA_TOKEN_FILE = "/var/run/secrets/kubernetes.io/serviceaccount/token"

if os.path.isfile(SA_TOKEN_FILE):
    with open(SA_TOKEN_FILE) as f:
        TOKEN = f.read().strip()
    print("Using mounted ServiceAccount token")
else:
    TOKEN = subprocess.run(
        ["oc", "whoami", "-t"], capture_output=True, text=True
    ).stdout.strip()
    print("Using oc whoami -t token")

os.environ["MLFLOW_TRACKING_TOKEN"] = TOKEN
print("Token set:", TOKEN[:20], "...")

## Step 3 — Set your MLflow tracking URL

Replace the URL below with the MLflow server URL provided by your instructor.

In [ ]:
import mlflow

MLFLOW_TRACKING_URI = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("my-first-experiment")

print("Tracking URI:", mlflow.get_tracking_uri())

## Step 4 — Trace an AI model call

We enable `mlflow.openai.autolog()` so every call to the model is automatically recorded as a trace in MLflow — no manual logging needed.

In [ ]:
import base64
from openai import OpenAI

# get the MaaS API token from the secret
result = subprocess.run(
    ["oc", "get", "secret", "maas-secret", "-o", "jsonpath={.data.token}"],
    capture_output=True, text=True
)
MAAS_TOKEN = base64.b64decode(result.stdout.strip()).decode()

# enable automatic tracing — every OpenAI call is recorded in MLflow
mlflow.openai.autolog()

client = OpenAI(
    base_url="https://maas.apps.ocp.cloud.rhai-tmm.dev/prelude-maas/qwen36-27b/v1",
    api_key=MAAS_TOKEN
)

with mlflow.start_run():
    response = client.chat.completions.create(
        model="qwen36-27b",
        messages=[{"role": "user", "content": "In one sentence, what is MLflow?"}]
    )
    print(response.choices[0].message.content)
    print("\nOpen the MLflow UI to see the full trace.")